In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/cumin_all_states_raw.csv', parse_dates=['date'])
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (48141, 12)


,date,state,market,variety,grade,arrivals,unit_of_arrivals,min_price,max_price,modal_price,unit_of_price,commodity
0,2020-01-01,Gujarat,Amreli APMC,Cummin Seed(Jeera),FAQ,0.60,Metric Tonnes,13750.0,14550.0,13925.0,Rs./Quintal,Cumin (Jeera)
1,2020-01-01,Gujarat,Bhabhar APMC,Other,FAQ,0.80,Metric Tonnes,13450.0,14310.0,13880.0,Rs./Quintal,Cumin (Jeera)
2,2020-01-01,Gujarat,Bhavnagar APMC,Other,FAQ,0.20,Metric Tonnes,11500.0,13000.0,12250.0,Rs./Quintal,Cumin (Jeera)
3,2020-01-01,Gujarat,Botad APMC,Other,FAQ,4.30,Metric Tonnes,11280.0,15680.0,13480.0,Rs./Quintal,Cumin (Jeera)
4,2020-01-01,Gujarat,Dasada Patadi APMC,Cummin Seed(Jeera),FAQ,0.48,Metric Tonnes,13300.0,14455.0,13877.5,Rs./Quintal,Cumin (Jeera)


In [2]:
# From your EDA: upper IQR fence = ₹37,538
# We CAP (not drop) so 2023 spike is preserved as a real market event
UPPER_FENCE = 37538

df['modal_price_capped'] = df['modal_price'].clip(upper=UPPER_FENCE)
df['min_price_capped']   = df['min_price'].clip(upper=UPPER_FENCE)
df['max_price_capped']   = df['max_price'].clip(upper=UPPER_FENCE)

print(f"Outliers capped: {(df['modal_price'] > UPPER_FENCE).sum()} rows")

Outliers capped: 3133 rows


In [3]:
# Weighted average price (weighted by arrivals), sum arrivals
agg = df.groupby(['date', 'state']).agg(
    modal_price   = ('modal_price_capped', lambda x: np.average(x, weights=df.loc[x.index, 'arrivals'])),
    arrivals      = ('arrivals', 'sum'),
    num_markets   = ('market', 'nunique')
).reset_index()

print(f"Aggregated shape: {agg.shape}")
agg.head()

Aggregated shape: (6045, 5)


,date,state,modal_price,arrivals,num_markets
0,2020-01-01,Gujarat,13940.425455,464.82,17
1,2020-01-01,Maharashtra,21000.000000,2.00,1
2,2020-01-01,Rajasthan,14000.000000,33.10,2
3,2020-01-02,Gujarat,14273.604281,687.18,26
4,2020-01-02,Rajasthan,14679.367089,23.70,2


In [4]:
agg['year']             = agg['date'].dt.year
agg['month']            = agg['date'].dt.month
agg['quarter']          = agg['date'].dt.quarter
agg['day_of_week']      = agg['date'].dt.dayofweek
agg['day_of_year']      = agg['date'].dt.dayofyear
agg['is_harvest_season']= agg['month'].isin([2, 3, 4]).astype(int)  # Feb-Apr peak from EDA
agg['is_lean_season']   = agg['month'].isin([5, 6, 7, 8]).astype(int)  # May-Aug high price

In [5]:
agg = agg.sort_values(['state', 'date']).reset_index(drop=True)

for state, grp in agg.groupby('state'):
    idx = grp.index
    # Lag features
    agg.loc[idx, 'price_lag_7']    = grp['modal_price'].shift(7)
    agg.loc[idx, 'price_lag_14']   = grp['modal_price'].shift(14)
    agg.loc[idx, 'price_lag_30']   = grp['modal_price'].shift(30)
    agg.loc[idx, 'arrivals_lag_7'] = grp['arrivals'].shift(7)
    # Rolling averages
    agg.loc[idx, 'price_roll_7']    = grp['modal_price'].rolling(7,  min_periods=1).mean()
    agg.loc[idx, 'price_roll_30']   = grp['modal_price'].rolling(30, min_periods=1).mean()
    agg.loc[idx, 'arrivals_roll_7'] = grp['arrivals'].rolling(7,     min_periods=1).mean()

print(f"NaNs from lags: {agg.isnull().sum().sum()}")
print(f"Final shape: {agg.shape}")

NaNs from lags: 492
Final shape: (6045, 19)


In [6]:
agg.to_csv('../data/processed/cumin_processed.csv', index=False)
print("Saved to data/processed/cumin_processed.csv")
print(agg.describe())

Saved to data/processed/cumin_processed.csv
                                date   modal_price      arrivals  num_markets  \
count                           6045   6045.000000   6045.000000  6045.000000   
mean   2022-12-20 21:43:44.516129024  20893.866255    323.545500     7.958644   
min              2020-01-01 00:00:00   1000.000000      0.010000     1.000000   
25%              2021-06-09 00:00:00  13791.684403      5.750000     1.000000   
50%              2022-12-15 00:00:00  20000.000000     45.000000     3.000000   
75%              2024-06-30 00:00:00  25000.000000    387.800000    13.000000   
max              2025-12-31 00:00:00  37538.000000  10473.870000    44.000000   
std                              NaN   8215.628999    671.165675     9.163253   

              year        month      quarter  day_of_week  day_of_year  \
count  6045.000000  6045.000000  6045.000000  6045.000000  6045.000000   
mean   2022.483044     6.369727     2.452771     2.656907   178.597849   
min 